In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
import matplotlib.animation as animation
import matplotlib.colors as colors
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 2**128

from IPython.display import HTML
from astropy.stats import sigma_clip
from tesscube import TESSCube
from astropy.time import Time
from tqdm import tqdm

DATADIR = "/Users/jimartin/Work/TESS/tess-asteroid-ml/data"

# We open the track file

In [ ]:
sector = 7
camera = 3
ccd = 4

In [ ]:
track = pd.read_csv(f"{DATADIR}/pred_asteroid_tracks/sector{sector:04}/{camera}-{ccd}/asteroid_positions_s{sector:04}-{camera}-{ccd}_001.csv")
track.rename(columns={"Time (JD)": "mjd"}, inplace=True)
X = track["X"].values.copy()
Y = track["Y"].values.copy()
mask = track["Flag"] == "F"
X[mask] = track["Y"][mask].values.copy()
Y[mask] = track["X"][mask].values.copy()
track["X"] = X
track["Y"] = Y
track.head(20)

In [ ]:
track.info()

In [ ]:
# plt.scatter(track.X_fit, track.Y_fit, s=1)
plt.title(f"Asteroid Track in Sector {sector} Camera {camera} CCD {ccd}")
plt.scatter(track.query("Flag == 'P'").X, track.query("Flag == 'P'").Y, s=2, c="tab:red", label="Projected")
plt.scatter(track.query("Flag == 'F'").X, track.query("Flag == 'F'").Y, s=2, c="tab:blue", label="Detected")
plt.xlabel("Column")
plt.ylabel("Row")
plt.legend()
plt.show()

# Fix cutout coverint the entire track

This solution is good for short tracks that cover relatively small region of the TPF (200 pix^2).
For longer tracks, this is ineficient. But it has the adventage to make background (background level + stars) modeling easier and robust.

In [ ]:
# we find the corner and size of the cutout from the track

corner = np.floor((np.maximum(0, track.Y_fit.min() - 30), np.maximum(44, track.X_fit.min() - 30))).astype(int)
size = np.ceil((np.minimum(2048, track.Y_fit.max() + 30), np.minimum(2096, track.X_fit.max() + 30))).astype(int)
size -= corner
print(corner)
print(size)

In [ ]:
rips = TESSCube(sector=sector, camera=camera, ccd=ccd)

In [ ]:
# download only pixels that enclose the track 

flx, flx_e = rips.get_flux(shape=tuple(size), corner=tuple(corner))
flx.shape, flx_e.shape

In [ ]:
row, col = np.mgrid[corner[0] : corner[0] + size[0], corner[1] : corner[1] + size[1]]
row.shape, col.shape

In [ ]:
rips.cadence_number.shape, rips.quality.shape, rips.time.shape

In [ ]:
# fix time units
ffi_time = Time(rips.time + 2457000, format="mjd", scale="tdb").mjd

In [ ]:
np.median(rips.last_hdu.data["TELAPSE"])

We need to align the track and FFI times. This shouldn't be necessary in the future, as tracks will include cadence number, making easier the alignment. We also need to make sure the cadence number and the FFI index are right

In [ ]:
# find track cadence number
track_time = np.concatenate([[ffi_time[0]], track.mjd])
track_cad = np.cumsum(
            np.round(np.diff(track_time) / np.median(rips.last_hdu.data["TELAPSE"]))
            ).astype(int)
track["cadenceno"] = track_cad

# correct track mjd
# track["mjd"] -= track.mjd.diff().median() #* np.diff(track.cadenceno)

track["ffi_index"] = np.array([np.where(np.isclose(d, ffi_time, atol=0.01, rtol=0))[0] for d in track.mjd]).ravel() - 1
track["quality"] = rips.quality[track.ffi_index.values].astype(int)
track

Make sure we are using frames with good quality.

In [ ]:
track.quality.value_counts()

In [ ]:
# we make a "static" frame with a median stack, we reject bad cadences

to_stack = sigma_clip(flx[track.query("quality == 0")["ffi_index"].values], axis=0, sigma=3, cenfunc="median", masked=False)
statics = np.median(to_stack, axis=0)
statics = np.median(flx[track["ffi_index"].values], axis=0)
statics.shape

In [ ]:
def animate_static(col, row, data, track,
                    x_axis_label="X", 
                    y_axis_label="Y", interval=200, repeat_delay=1000):
    """
    Animates 2D input data as a function of track.mjd.values.

    Args:
        data (numpy.ndarray): The 2D input data.
        x_axis_label (str, optional): The label for the x-axis. Defaults to "X".
        y_axis_label (str, optional): The label for the y-axis. Defaults to "Y".
        interval (int, optional): Delay between frames in milliseconds. Defaults to 20.
        repeat_delay (int, optional): Delay before repeating the animation in milliseconds. Defaults to 1000.
    """

    vlo, lo, mid, hi, vhi = np.nanpercentile(data, [0.2, 1, 50, 95, 99.9])
    # print(vlo, lo, mid, hi, vhi)
    try:
        cnorm = colors.LogNorm(vmin=vlo, vmax=vhi)
    except ValueError:
        cnorm = colors.Normalize(vmin=vlo, vmax=vhi)

    fig, ax = plt.subplots()
    fig.suptitle(f"Asteroid Tracks in Sector {sector} Camera {camera} CCD {ccd}")
    im = ax.pcolormesh(
        col, row,
        data[0], 
        cmap="Greys_r",
        vmin=lo, 
        vmax=vhi,
        # norm=cnorm,
        rasterized=True
    )
    line = ax.plot(track.X_fit.values[0], track.Y_fit.values[0], ".-", c="tab:red", alpha=.5, lw=1)[0]
    ax.axis('equal')
    ax.set_title(f"CAD {track.cadenceno.values[0]} | BTJD {track.mjd.values[0] - 2457000:.4f}")
    plt.colorbar(im, location="right", shrink=.8, label="Flux [-e/s]")

    def animate(i):
        im.set_array(data[i].ravel())
        line.set_xdata(track.X_fit.values[:i])
        line.set_ydata(track.Y_fit.values[:i])
        ax.set_title(f"CAD {track.cadenceno.values[i]} | BTJD {track.mjd.values[i] - 2457000:.4f}")
        return (im, line, )

    plt.close(ax.figure)

    # Create the animation
    ani = animation.FuncAnimation(
        fig, animate, frames=len(data), interval=interval, blit=True, repeat_delay=repeat_delay, repeat=True,
    )

    ax.set_xlabel(x_axis_label)
    ax.set_ylabel(y_axis_label)

    # ax.plot(track.X_fit, track.Y_fit, c="tab:red", alpha=.5, lw=1)

    return ani

We can do aperture photometry in this image difference.
We also need a better background + static estimation. 
A good solution is using a rolling median window (50 frames), the caveat is frames at the beggining and end of the sector will be poorly extimated. 

In [ ]:
track_ = track.query("quality == 0")
HTML(animate_static(
    col, 
    row, 
    flx[track_.ffi_index.values] - statics, 
    track_
).to_jshtml())

**For predicted tracks from the ML model:**

the tracks seem to be shorter that the true, meaning there are missing predictions before the start and after the end of the predicted track. This could be because the NN takes time to "warm up" to high probabilities aftr the asteroid enters the scene. Possible (naive) solution is to extrapolate the positions from the predicted tracks to ~10 frames before and after. We need to find somethign more robust. 

In [ ]:
# naive rolling median window in time
# this might not be efficient
def build_static(cube, window=50):
    nt, nx, ny = cube.shape
    df = pd.DataFrame(cube.reshape(nt, nx*ny))
    rmed = df.rolling(window, min_periods=10, center=True, closed="both").median().values.reshape(nt, nx, ny)
    # silly solution of repeating the first 50 frames to match the shape
    # this needs to be better, maybe interpolation?
    # med = np.median(cube[:window], axis=0)
    # rmed[:window-1] = np.repeat(med.reshape(-1, nx, ny), window-1, axis=0)
    return rmed

In [ ]:
static_mod_pd = build_static(flx, window=101)
static_mod_pd.shape

This is better, the noise statistics and artifacts from bad substraction are improved.

In [ ]:
track_ = track.query("quality == 0")
HTML(animate_static(
    col, 
    row, 
    flx[track_.ffi_index.values] - static_mod_pd[track_.ffi_index.values], 
    track=track_,
).to_jshtml())

## LC

In [ ]:
from scipy import stats
from matplotlib import patches
from scipy import signal, ndimage

In [ ]:
static_tpf = flx[track.ffi_index.values] - static_mod_pd[track.ffi_index.values]

In [ ]:
def conv2d(a, f):
    s = f.shape + tuple(np.subtract(a.shape, f.shape) + 1)
    strd = np.lib.stride_tricks.as_strided
    subM = strd(a, shape = s, strides = a.strides * 2)
    return np.einsum('ij,ijkl->kl', f, subM)

def remove_isolated_pixels(mask):
    """Removes isolated pixels from a 2D mask.
    
    Args:
    mask: A 2D numpy array representing the mask.
    
    Returns:
    A new 2D numpy array with isolated pixels removed.
    """
    
    # Create a kernel to detect isolated pixels
    kernel = np.array([[0, 1, 0],
                       [1, 1, 1],
                       [0, 1, 0]])
    
    # Apply the kernel to the mask
    clean_mask = np.logical_and(mask, (signal.convolve2d(mask, kernel, mode="same") != 0))
    
    return clean_mask

In [ ]:
nt = 2
ap_rad = 6
aperture_mask = np.zeros_like(static_tpf).astype(bool)

# iterate over all frames
for nt in range(len(static_tpf)-1):

    # 1st rough circular mask of rad 6 pix around the pred position
    circ_ap = np.hypot(col - track.X_fit.values[nt], row - track.Y_fit.values[nt]) <= ap_rad
    # mask with value threshold 
    median, mad = np.median(static_tpf[nt + 1]), stats.median_abs_deviation(static_tpf[nt + 1].ravel())
    thresh_mask = static_tpf[nt + 1][circ_ap] >= median + 3 * mad

    # combine circular and threshold masks
    for c, r in zip(col[circ_ap][thresh_mask], row[circ_ap][thresh_mask]):
        aperture_mask[nt + 1][(col == c) & (row == r)] = True

    # refine mask to remove isolated pixels and open borders.. 
    # aperture_mask[nt + 1] = remove_isolated_pixels(aperture_mask[nt + 1])
    # aperture_mask[nt + 1] = ndimage.binary_closing(aperture_mask[nt + 1])
    aperture_mask[nt + 1] = ndimage.binary_erosion(aperture_mask[nt + 1])
    aperture_mask[nt + 1] = ndimage.binary_dilation(aperture_mask[nt + 1])

# circ_ap.sum(), thresh_mask.sum(), aperture_mask.sum()

sap = np.array([[np.nansum(x[m]), np.nansum(x[m] ** 2) ** 0.5] for x, m in zip(static_tpf, aperture_mask)])

track["sap_flux"] = sap[:,0]
track["sap_flux_err"] = sap[:,1]

In [ ]:
track.head(20)

In [ ]:
vlo, lo, mid, hi, vhi = np.nanpercentile(static_tpf, [0.2, 1, 50, 95, 99.8])

def plot_img_aperture(col, row, img, aperture_mask, ax=None, radius=6, size=15,
                      xy=[0,0], nt=0, cbar=False,
                      x_axis_label="Column", 
                      y_axis_label="Row", **kwargs):

    if ax is None:
        fig, ax = plt.subplots()
        fig.suptitle(f"Asteroid Tracks in Sector {sector} Camera {camera} CCD {ccd}")
    im = ax.pcolormesh(
        col, row,
        img, 
        cmap="Greys_r",
        vmin=lo, 
        vmax=vhi,
        # norm=cnorm,
        rasterized=True
    )
    dot = ax.scatter(xy[0], xy[1], marker="x", c="gold", alpha=1, s=100)
    ax.axis('equal')
    ax.set_title(f"CAD {track.cadenceno.values[nt]} | BTJD {track.mjd.values[nt] - 2457000:.4f}")
    if cbar:
        plt.colorbar(im, location="right", shrink=.8, label="Flux [-e/s]")
    
    circle = plt.Circle((xy[0], xy[1]), radius=radius, fill=False, lw=1, color="tab:red")
    ax.add_patch(circle)
    
    for i, pi in enumerate(row[:, 0]):
        for j, pj in enumerate(col[0, :]):
            if aperture_mask[i, j]:
                # print("here")
                rect = patches.Rectangle(
                    xy=(pj - 0.5, pi - 0.5),
                    width=1,
                    height=1,
                    color="tab:red",
                    fill=False,
                    # hatch="//",
                    alpha=0.8,
                )
                ax.add_patch(rect)
    
    ax.set_xlim(xy[0] - size, xy[0] + size)
    ax.set_ylim(xy[1] - size, xy[1] + size)

    ax.set_xlabel(x_axis_label)
    ax.set_ylabel(y_axis_label)

    # ax.plot(track.column, track.row, c="tab:red", alpha=.5, lw=1)

    return ax

def animate_image(col, row, cube, aperture_mask, 
                  radius=6, size=15,
                  interval=200, repeat_delay=1000):

    fig, ax = plt.subplots()
    fig.suptitle(f"Asteroid Tracks in Sector {sector} Camera {camera} CCD {ccd}")
    nt = 10

    ax = plot_img_aperture(col, row, cube[nt+1], aperture_mask[nt+1], ax=ax, radius=6, size=15,
                           xy=[track.X_fit.values[nt], track.Y_fit.values[nt]],
                      x_axis_label="Column", cbar=True,
                      y_axis_label="Row")

    def animate(nt):
        ax.clear()
        _ = plot_img_aperture(col, row, cube[nt+1], aperture_mask[nt+1], ax=ax, radius=6, size=15,
                               xy=[track.X_fit.values[nt], track.Y_fit.values[nt]], nt=nt, cbar=False,
                               x_axis_label="Column", y_axis_label="Row", 
                               vlo=vlo, lo=lo, mid=mid, hi=hi, vhi=vhi)
        
        return ()

    plt.close(ax.figure)

    # Create the animation
    ani = animation.FuncAnimation(
        fig, animate, frames=len(cube) - 1, interval=interval, blit=True, repeat_delay=repeat_delay, repeat=True,
    )

    return ani
    

In [ ]:
HTML(
    animate_image(col, row, static_tpf, aperture_mask, radius=6, size=15).to_jshtml()
)

In [ ]:
# save animation to movie file
animate_image(col, row, static_tpf, aperture_mask, radius=6, size=15).save("./data/asteroid_001_anim_mov.gif", writer="pillow")

In [ ]:
plt.figure(figsize=(12,3))
plt.title("Asteroid Light Curve")
proj = track.query("Flag == 'P'")
plt.errorbar(proj.mjd - 2457000, proj.sap_flux, yerr=proj.sap_flux_err, fmt=".", c="tab:red", label="Projected")
fitted = track.query("Flag == 'F'")
plt.errorbar(fitted.mjd - 2457000, fitted.sap_flux, yerr=fitted.sap_flux_err, fmt=".", c="tab:blue", label="Detected")
plt.xlabel("Time [TBJD]")
plt.ylabel("Flux [-e/s]")
plt.legend()
plt.show()

In [ ]:
import lkprf


In [ ]:
prf = lkprf.TESSPRF(camera=camera, ccd=ccd)

In [ ]:
tuple(size)

In [ ]:
track.query("Flag == 'F'")

In [ ]:
prf1 = prf.evaluate(targets=[(1889.05, 109.48), (1882.38, 119.32)], origin=tuple(corner), shape=tuple(size))
prf1.shape

In [ ]:
plt.pcolormesh(
        col, row,
        prf1[0], 
        cmap="Greys_r",
        vmin=0, 
        vmax=0.2,
        rasterized=True
    )
plt.xlim(109.48 - 7, 109.48 + 7)
plt.ylim(1889.05 - 7, 1889.05 + 7)
plt.show()

In [ ]:
plt.pcolormesh(
        col, row,
        prf1[1], 
        cmap="Greys_r",
        vmin=0, 
        vmax=0.2,
        rasterized=True
    )
plt.xlim(119.32 - 7, 119.32 + 7)
plt.ylim(1882.38 - 7, 1882.38 + 7)
plt.show()

# Moving TPF

Now let's make a moving cutout around the predicted trajectory using the track positions and `tessrip`.
Modeling the background is more difficult because the background is always changing. A solution might be downloading a slightly larger cutout around the positions in ~50 frames, and use those to model the background (rolling median window of 50 frames). But need to make sure we match the pixel positions and times the right way.

In [ ]:
cutout_size = 50
moving_flux = np.empty((len(track), cutout_size, cutout_size))
moving_flux_e = np.empty((len(track), cutout_size, cutout_size))
moving_row = np.empty((len(track), cutout_size, cutout_size), dtype=int)
moving_col = np.empty((len(track), cutout_size, cutout_size), dtype=int)

# we iter over track positions to get the moving tpf
for k, r in tqdm(track.iterrows(), total=len(track)):
    # print(k, r)
    corner = np.floor(r.r - cutout_size / 2).astype(int), np.floor(r.column - cutout_size / 2).astype(int)
    # print(corner, (cutout_size, cutout_size), (int(r.ffi_index), int(r.ffi_index + 1)))
    
    frame, frame_e = rips.get_flux(shape=(cutout_size, cutout_size), 
                               corner=tuple(corner),
                               frame_range=(int(r.ffi_index), int(r.ffi_index + 1)),
                              )
    moving_flux[k] = frame.array
    moving_flux_e[k] = frame_e.array

    moving_row[k], moving_col[k] = np.mgrid[corner[0] : corner[0] + cutout_size, corner[1] : corner[1] + cutout_size]
    # break

In [ ]:
def animate_2d_data(data,
                    x_axis_label="Column", 
                    y_axis_label="Row", interval=200, repeat_delay=1000):

    vlo, lo, mid, hi, vhi = np.nanpercentile(data, [0.2, 1, 50, 95, 99.8])
    # print(vlo, lo, mid, hi, vhi)
    try:
        cnorm = colors.LogNorm(vmin=vlo, vmax=vhi)
    except ValueError:
        cnorm = colors.Normalize(vmin=vlo, vmax=vhi)

    fig, ax = plt.subplots()
    fig.suptitle(f"Asteroid Tracks in Sector {sector} Camera {camera} CCD {ccd}")
    im = ax.imshow(
        data[0], 
        cmap="Greys_r",
        vmin=lo, 
        vmax=mid + 50,
        # norm=cnorm,
        rasterized=True,
        origin="lower",
    )
    ax.axis('equal')
    # ax.set_title(f"CAD {track.cadenceno.values[0]} | BTJD {track.mjd.values[0] - 2457000:.4f}")
    plt.colorbar(im, location="right", shrink=.9, label="Flux [-e/s]")

    def animate(i):
        im.set_array(data[i])
        # ax.set_title(f"CAD {track.cadenceno.values[i]} | BTJD {track.mjd.values[i] - 2457000:.4f}")
        return (im, )

    plt.close(ax.figure)

    # Create the animation
    ani = animation.FuncAnimation(
        fig, animate, frames=len(data), interval=interval, blit=True, repeat_delay=repeat_delay, repeat=True,
    )

    ax.set_xlabel(x_axis_label)
    ax.set_ylabel(y_axis_label)

    # ax.plot(track.column, track.row, c="tab:red", alpha=.5, lw=1)

    return ani

Really hard to see the asteroid in the raw data. This needs image subtraction, but hard to model the background when is moving.

In [ ]:
HTML(animate_2d_data(moving_flux).to_jshtml())